# ReviewIQ — NLP Pipeline (Prototype: Netflix)

Prototyping the pipeline on **one app (Netflix)** with a 10k sample, then scaling.

Flow: embeddings → clustering → label clusters → sentiment → LLM summary.

Reads `../data/processed/master_clean_lang.parquet`. No language filter (see HANDOFF §7).

## Stage 1 — Embeddings

In [6]:
import pandas as pd
from pathlib import Path

# Read the language-tagged clean data
reviews = pd.read_parquet("../data/processed/master_clean_lang.parquet")

# Prototype on ONE app
netflix = reviews[reviews["app_name"] == "netflix"].copy()

# Embedding gate: keep reviews with content length >= 10 (decision #4).
# NOTE: NO language filter here — we keep all languages (HANDOFF §7).
netflix["char_len"] = netflix["content"].str.len()
netflix = netflix[netflix["char_len"] >= 10]
print("Netflix reviews after length gate:", len(netflix))

# For a FAST first pass, work on a random 10k sample.
# Later, set SAMPLE_SIZE = None to run the full Netflix set.
SAMPLE_SIZE = 10_000
if SAMPLE_SIZE:
    netflix = netflix.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    netflix = netflix.reset_index(drop=True)

print("Working set:", len(netflix))

Netflix reviews after length gate: 93064
Working set: 10000


In [7]:
from sentence_transformers import SentenceTransformer

# First run downloads ~470 MB, then it's cached on disk for next time.
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model loaded. Vector size:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded. Vector size: 384


C:\Users\nithi\AppData\Local\Temp\ipykernel_23388\1835242314.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Vector size:", model.get_sentence_embedding_dimension())


In [8]:
import numpy as np

texts = netflix["content"].tolist()

# Turn every review into a vector. The progress bar lets you watch it work.
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print("Embeddings shape:", embeddings.shape)   # (10000, 384)

# Save the vectors AND the matching reviews together, so they stay aligned.
out = Path("../data/processed/embeddings")
out.mkdir(parents=True, exist_ok=True)
np.save(out / "netflix_sample.npy", embeddings)
netflix.to_parquet(out / "netflix_sample.parquet", index=False)
print("Saved:", out)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (10000, 384)
Saved: ..\data\processed\embeddings


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

i = 0  # try changing this to any row number
sims = cosine_similarity(embeddings[i:i+1], embeddings)[0]
top = sims.argsort()[::-1][1:6]   # the 5 most similar reviews (skipping itself)

print("QUERY:", texts[i])
print("\nMost similar reviews:")
for j in top:
    print(f"  ({sims[j]:.2f})  {texts[j][:100]}")

QUERY: I like it because I can watch videos

Most similar reviews:
  (0.76)  I like this because it's full of amazing movies, shows, and more I love it
  (0.76)  i love it i can watch all my movies and shows on the go.
  (0.74)  I like it very much
  (0.74)  I love it. I can watch all of my shows and catch up on things
  (0.74)  I Like it very much


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

emb_dir = Path("../data/processed/embeddings")
embeddings = np.load(emb_dir / "netflix_sample.npy")
netflix = pd.read_parquet(emb_dir / "netflix_sample.parquet")

print("embeddings:", embeddings.shape, "| reviews:", netflix.shape)

embeddings: (10000, 384) | reviews: (10000, 11)


In [11]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,       # how much local vs global structure to preserve
    n_components=5,       # squash 384 dimensions down to 5
    metric="cosine",      # compare vectors the same way embeddings are meant to be compared
    random_state=42,      # reproducible
    init="random",  
)
embeddings_5d = reducer.fit_transform(embeddings)
print("reduced shape:", embeddings_5d.shape)   # (10000, 5)

c:\Users\nithi\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


reduced shape: (10000, 5)


In [12]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,   # a group must have >= 30 reviews to count as a topic
    min_samples=5,        # how conservative to be (higher = more points called noise)
    metric="euclidean",
)
labels = clusterer.fit_predict(embeddings_5d)
netflix["cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels == -1).sum()
print("clusters found:", n_clusters)
print(f"noise (unclustered): {noise}  ({noise/len(labels)*100:.1f}%)")
print("\ncluster sizes:")
print(netflix["cluster"].value_counts().sort_index())

clusters found: 31
noise (unclustered): 2567  (25.7%)

cluster sizes:
cluster
-1     2567
 0      115
 1     2947
 2       75
 3       86
 4       46
 5      164
 6       32
 7       36
 8       32
 9       40
 10      71
 11     112
 12      33
 13     817
 14      65
 15     113
 16      38
 17      99
 18     145
 19     144
 20     118
 21      74
 22      74
 23      30
 24     117
 25     847
 26     414
 27      47
 28     390
 29      47
 30      65
Name: count, dtype: int64


In [13]:
for c in sorted(netflix["cluster"].unique()):
    if c == -1:
        continue  # skip the noise bucket
    subset = netflix[netflix["cluster"] == c]
    print(f"\n=== cluster {c}  (n={len(subset)}) ===")
    for t in subset["content"].head(5).tolist():
        print("  -", t[:100])


=== cluster 0  (n=115) ===
  - EDIT 2023/11/18: I have control of brightness again. Good, but I can't remove the game icons from th
  - Samsung Tablet S8+ I am using the most recent Netflix app Running into issue where the brightness fo
  - An update this year has made this app's brightness blinding when trying to watch in the dark. Even m
  - Last week i can adjust the brightness while watching movies on netflix. But now, the brightness leve
  - The brightness controls are useless.

=== cluster 1  (n=2947) ===
  - I love Netflix a lot because you can watch any movie you want.
  - it is always breaking down you need to fix it.it tells me my device is not working. that is bull bec
  - I would like to have more options for the Turkish Netflix also it would be nice if new seasons and n
  - Love netflix I have no words great binge watching app for me mabey you too
  - I like to watch movies therefore when I used Netflix I can watch many movies and dramas in HD versio

=== cluster 2  (n=75

In [14]:
# Persist the cluster assignments so this work survives a kernel restart
netflix.to_parquet("../data/processed/embeddings/netflix_clustered.parquet", index=False)
print("saved:", netflix.shape)

saved: (10000, 12)


In [15]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

def top_words(texts, n=6):
    # Count words across all reviews in one cluster, ignoring English filler
    # words ("the", "is") and words seen in fewer than 2 reviews (typos).
    vec = CountVectorizer(stop_words="english", min_df=2)
    try:
        X = vec.fit_transform(texts)
    except ValueError:          # cluster too small/empty for the vectorizer
        return []
    totals = np.asarray(X.sum(axis=0)).ravel()
    vocab = vec.get_feature_names_out()
    return [vocab[i] for i in totals.argsort()[::-1][:n]]

In [16]:
rows = []
for c in sorted(netflix["cluster"].unique()):
    if c == -1:
        continue                       # skip noise
    sub = netflix[netflix["cluster"] == c]
    rows.append({
        "cluster": c,
        "n_reviews": len(sub),
        "avg_score": round(sub["score"].mean(), 1),
        "top_words": ", ".join(top_words(sub["content"].tolist())),
        "example": sub["content"].iloc[0][:80],
    })

topics = pd.DataFrame(rows).sort_values("n_reviews", ascending=False)

# Show the whole table, widest column included
pd.set_option("display.max_colwidth", 60)
topics

,cluster,n_reviews,avg_score,top_words,example
1,1,2947,2.7,"netflix, app, watch, movies, shows, love",I love Netflix a lot because you can watch any movie you...
25,25,847,3.7,"movies, shows, watch, good, like, love",I like it because I can watch videos
13,13,817,4.4,"app, movies, good, watch, love, shows","Good result voice clear, brilliant app"
26,26,414,2.0,"video, app, play, audio, screen, watch",It is OK but when used with chromecast device delayed vo...
28,28,390,1.6,"app, open, error, working, update, phone",Horrible! I cannot delete this app from my device!!! It ...
5,5,164,2.9,"movies, hindi, english, language, languages, app",Translate all shows in hindi many of them are in differe...
18,18,145,3.8,"hai, hi, ka, bhi, ko, lo",फैंटास्टिक
19,19,144,4.8,"good, nice, great, experience, perfect, super",you need too add gold rush discovery channel version and...
20,20,118,1.5,"password, account, sign, login, email, tried",Trouble login in
24,24,117,1.5,"payment, card, account, money, pay, don",They took money from my debit card as a subscription fee...


In [17]:
topics.to_parquet("../data/processed/embeddings/netflix_topics.parquet", index=False)
topics.to_csv("../data/processed/embeddings/netflix_topics.csv", index=False)
print("saved topic table:", topics.shape)

saved topic table: (31, 5)


In [18]:
from transformers import pipeline

# Multilingual sentiment model: returns negative / neutral / positive per text.
# First run downloads ~1.1 GB, then it's cached like the embedding model.
sentiment = pipeline(
    "sentiment-analysis",
       model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    truncation=True,     # reviews longer than the model's limit get cut, not crash
)
print(sentiment("This app keeps crashing and support ignores me"))
print(sentiment("Absolutely love it, best streaming service ever"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.39474454522132874}]
[{'label': 'positive', 'score': 0.9909538626670837}]


In [19]:
texts = netflix["content"].tolist()

# Process in batches; this is the 10-15 minute cell. Progress prints every 1000.
results = []
BATCH = 64
for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    results.extend(sentiment(batch, batch_size=BATCH))
    if (i // BATCH) % 16 == 0:
        print(f"{i}/{len(texts)}")

netflix["sentiment"] = [r["label"] for r in results]
netflix["sentiment_conf"] = [round(r["score"], 3) for r in results]
print(netflix["sentiment"].value_counts())

0/10000
1024/10000
2048/10000
3072/10000
4096/10000
5120/10000
6144/10000
7168/10000
8192/10000
9216/10000
sentiment
negative    4652
positive    4540
neutral      808
Name: count, dtype: int64


In [20]:
netflix.to_parquet("../data/processed/embeddings/netflix_clustered.parquet", index=False)

# Do text-sentiment and star ratings agree? Cross-tabulate them.
print(pd.crosstab(netflix["score"], netflix["sentiment"], normalize="index").round(2))

sentiment  negative  neutral  positive
score                                 
1              0.73     0.11      0.16
2              0.67     0.12      0.21
3              0.51     0.11      0.38
4              0.26     0.06      0.68
5              0.09     0.03      0.88


In [21]:
import pandas as pd

reviews = pd.read_parquet("../data/processed/embeddings/netflix_clustered.parquet")
topics = pd.read_parquet("../data/processed/embeddings/netflix_topics.parquet")

print("reviews:", reviews.shape, "| topics:", topics.shape)
print("columns:", list(topics.columns))

reviews: (10000, 14) | topics: (31, 5)
columns: ['cluster', 'n_reviews', 'avg_score', 'top_words', 'example']


In [22]:
# Rule 1: a PAIN POINT is a topic where people are clearly angry.
pain = topics[topics["avg_score"] <= 2.5].sort_values(["n_reviews"], ascending=False)

# Rule 2: a PRO is a topic where people are clearly happy.
pros = topics[topics["avg_score"] >= 4.0].sort_values("n_reviews", ascending=False)

# Rule 3: a FEATURE REQUEST topic is one where many reviews use asking-words.
REQUEST_WORDS = ["add", "please", "wish", "want", "should", "would be", "need"]

def request_share(cluster_id):
    texts = reviews.loc[reviews["cluster"] == cluster_id, "content"].str.lower()
    asks = texts.apply(lambda t: any(w in t for w in REQUEST_WORDS))
    return asks.mean()

topics["request_share"] = topics["cluster"].apply(request_share)
requests = topics[topics["request_share"] >= 0.5].sort_values("request_share", ascending=False)

print("pain points:", len(pain), "| pros:", len(pros), "| request-heavy topics:", len(requests))

pain points: 19 | pros: 6 | request-heavy topics: 2


In [25]:
n = len(reviews)
sent_pct = (reviews["sentiment"].value_counts(normalize=True) * 100).round(0)

lines = []
lines.append(f"# ReviewIQ Report — Netflix\n")
lines.append(f"*Based on {n:,} reviews analyzed*\n")

# --- Executive summary: the numbers that matter, in two sentences ---
top_pain = pain.head(3)
lines.append("## Executive summary\n")
lines.append(
    f"Of {n:,} reviews analyzed, {sent_pct.get('negative', 0):.0f}% are negative, "
    f"{sent_pct.get('positive', 0):.0f}% positive. "
    f"The biggest pain points are: "
    + "; ".join(f"**{r.top_words.split(',')[0]}** ({r.n_reviews} reviews, avg {r.avg_score}★)"
                for r in top_pain.itertuples())
    + "."
)

# --- Pain points, each with a real quote ---
lines.append("\n## Top pain points\n")
for r in pain.head(5).itertuples():
    quote = reviews.loc[(reviews["cluster"] == r.cluster) &
                        (reviews["sentiment"] == "negative"), "content"].iloc[0][:140]
    lines.append(f"- **{r.top_words}** — {r.n_reviews} reviews, avg {r.avg_score}★")
    lines.append(f"  > \"{quote}\"")

# --- What users love ---
lines.append("\n## What users love\n")
for r in pros.head(3).itertuples():
    lines.append(f"- **{r.top_words}** — {r.n_reviews} reviews, avg {r.avg_score}★")

# --- Feature requests ---
lines.append("\n## Feature requests\n")
for r in requests.head(3).itertuples():
    lines.append(f"- **{r.top_words}** — {r.request_share:.0%} of its {r.n_reviews} reviews are asking for something")

report = "\n".join(lines)
print(report)

# ReviewIQ Report — Netflix

*Based on 10,000 reviews analyzed*

## Executive summary

Of 10,000 reviews analyzed, 47% are negative, 45% positive. The biggest pain points are: **video** (414 reviews, avg 2.0★); **app** (390 reviews, avg 1.6★); **password** (118 reviews, avg 1.5★).

## Top pain points

- **video, app, play, audio, screen, watch** — 414 reviews, avg 2.0★
  > "Facing an issue where in everytime I fast forward a playing video, it freezes the video but the audio continues. And if I pause and okay it "
- **app, open, error, working, update, phone** — 390 reviews, avg 1.6★
  > "Horrible! I cannot delete this app from my device!!! It only allows me to disable!!! Unjust!!!"
- **password, account, sign, login, email, tried** — 118 reviews, avg 1.5★
  > "Trouble login in"
- **payment, card, account, money, pay, don** — 117 reviews, avg 1.5★
  > "They took money from my debit card as a subscription fee that I did not want any subscription. They are like thiefs!"
- **brightness, ap

In [26]:
from pathlib import Path

Path("../reports").mkdir(parents=True, exist_ok=True)
Path("../reports/netflix_report.md").write_text(report, encoding="utf-8")
print("saved -> reports/netflix_report.md")

saved -> reports/netflix_report.md
